# Power Consumption Prediction — Team Overfitters
Training Notebook: Ridge Regression vs. MLP Neural Network

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Set to False to skip training and load saved models
TRAIN = True

DATA_PATH = '../instructions_examples/Datasets/powerpredict.csv'
MODEL_DIR = 'models'
FIGURE_DIR = 'figures'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
print('Setup done.')

## 1. Data Loading & Exploration

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nTarget (power_consumption) statistics:')
print(df['power_consumption'].describe())

In [ ]:
# Visualisation 1: Distribution of target variable + feature correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Target distribution
axes[0].hist(df['power_consumption'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Power Consumption (MW)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Power Consumption')
axes[0].axvline(df['power_consumption'].mean(), color='red', linestyle='--', label=f'Mean: {df["power_consumption"].mean():.0f}')
axes[0].legend()

# Temperature vs power consumption (Bedrock as example city)
axes[1].scatter(df['Bedrock_t'], df['power_consumption'], alpha=0.1, s=5, color='steelblue')
axes[1].set_xlabel('Bedrock Temperature (K)')
axes[1].set_ylabel('Power Consumption (MW)')
axes[1].set_title('Temperature vs. Power Consumption (Bedrock)')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig1_data_overview.png', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

## 2. Preprocessing

In [ ]:
# Separate features and target
X = df.drop(columns=['power_consumption'])
y = df['power_consumption'].values

# Identify feature types
# Drop weather_description (redundant with weather_main, too many categories)
desc_cols = [c for c in X.columns if 'weather_description' in c]
X = X.drop(columns=desc_cols)

cat_cols = [c for c in X.columns if X[c].dtype == 'object']   # weather_main x5
num_cols = [c for c in X.columns if X[c].dtype != 'object']

print(f'Numeric features:     {len(num_cols)}')
print(f'Categorical features: {len(cat_cols)} -> one-hot encoded')
print(f'Dropped:              {len(desc_cols)} weather_description columns (redundant)')

In [ ]:
# Train / Validation split (80 / 20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train size: {len(X_train):,}  |  Validation size: {len(X_val):,}')

# Preprocessor: StandardScaler for numeric, OneHotEncoder for categorical
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

## 3. Method 1: Ridge Regression

Ridge regression minimises the penalised least-squares objective:
$$\hat{w} = \arg\min_w \|Xw - y\|^2 + \alpha \|w\|^2$$
The regularisation parameter $\alpha$ prevents overfitting by shrinking the weights.

In [ ]:
if TRAIN:
    # Random search over alpha (log-uniform between 1e-3 and 1e4)
    np.random.seed(42)
    alphas = np.logspace(-3, 4, 60)   # 60 values from 0.001 to 10000

    ridge_val_rmse = []
    ridge_train_rmse = []

    for alpha in alphas:
        pipe = Pipeline([
            ('pre', preprocessor),
            ('model', Ridge(alpha=alpha))
        ])
        pipe.fit(X_train, y_train)
        ridge_train_rmse.append(np.sqrt(mean_squared_error(y_train, pipe.predict(X_train))))
        ridge_val_rmse.append(np.sqrt(mean_squared_error(y_val, pipe.predict(X_val))))

    best_alpha_idx = np.argmin(ridge_val_rmse)
    best_alpha = alphas[best_alpha_idx]
    print(f'Best alpha: {best_alpha:.4f}')
    print(f'Best validation RMSE: {ridge_val_rmse[best_alpha_idx]:.2f}')

    # Refit with best alpha
    best_ridge = Pipeline([
        ('pre', preprocessor),
        ('model', Ridge(alpha=best_alpha))
    ])
    best_ridge.fit(X_train, y_train)
    joblib.dump(best_ridge, f'{MODEL_DIR}/ridge_model.pkl')
    joblib.dump({'alphas': alphas, 'train_rmse': ridge_train_rmse, 'val_rmse': ridge_val_rmse,
                 'best_alpha': best_alpha}, f'{MODEL_DIR}/ridge_search.pkl')
    print('Ridge model saved.')
else:
    best_ridge = joblib.load(f'{MODEL_DIR}/ridge_model.pkl')
    search = joblib.load(f'{MODEL_DIR}/ridge_search.pkl')
    alphas, ridge_train_rmse, ridge_val_rmse = search['alphas'], search['train_rmse'], search['val_rmse']
    best_alpha = search['best_alpha']
    print(f'Ridge model loaded. Best alpha: {best_alpha:.4f}')

In [ ]:
# Visualisation 2: Ridge hyperparameter search (RMSE vs alpha)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(alphas, ridge_train_rmse, label='Train RMSE', color='steelblue')
ax.semilogx(alphas, ridge_val_rmse, label='Validation RMSE', color='orange')
ax.axvline(best_alpha, color='red', linestyle='--', label=f'Best α = {best_alpha:.3f}')
ax.set_xlabel('Regularisation parameter α (log scale)')
ax.set_ylabel('RMSE (MW)')
ax.set_title('Ridge Regression: Hyperparameter Search')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig2_ridge_hyperparameter.png', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

## 4. Method 2: MLP Neural Network

A multi-layer perceptron (MLP) with one or two hidden layers, trained via backpropagation and gradient descent (Adam optimiser). The network learns non-linear feature interactions that linear regression cannot capture.

In [ ]:
if TRAIN:
    # Random hyperparameter search
    np.random.seed(42)
    N_TRIALS = 30

    hidden_options = [(64,), (128,), (256,), (64, 32), (128, 64), (256, 128), (128, 64, 32)]
    lr_options = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
    activation_options = ['relu', 'tanh']

    mlp_results = []

    # Prefit preprocessor once to speed up search
    preprocessor.fit(X_train)
    X_train_proc = preprocessor.transform(X_train)
    X_val_proc   = preprocessor.transform(X_val)

    for trial in range(N_TRIALS):
        hidden = hidden_options[np.random.randint(len(hidden_options))]
        lr     = lr_options[np.random.randint(len(lr_options))]
        act    = activation_options[np.random.randint(len(activation_options))]

        mlp = MLPRegressor(
            hidden_layer_sizes=hidden,
            activation=act,
            learning_rate_init=lr,
            max_iter=200,
            random_state=42,
            early_stopping=False
        )
        mlp.fit(X_train_proc, y_train)
        val_rmse = np.sqrt(mean_squared_error(y_val, mlp.predict(X_val_proc)))
        train_rmse = np.sqrt(mean_squared_error(y_train, mlp.predict(X_train_proc)))
        mlp_results.append({'hidden': hidden, 'lr': lr, 'act': act,
                             'val_rmse': val_rmse, 'train_rmse': train_rmse, 'model': mlp})
        if (trial + 1) % 10 == 0:
            print(f'Trial {trial+1}/{N_TRIALS} — best val RMSE so far: {min(r["val_rmse"] for r in mlp_results):.2f}')

    best_mlp_result = min(mlp_results, key=lambda r: r['val_rmse'])
    best_mlp_model  = best_mlp_result['model']

    print(f"\nBest MLP config:")
    print(f"  Hidden layers: {best_mlp_result['hidden']}")
    print(f"  Learning rate: {best_mlp_result['lr']}")
    print(f"  Activation:    {best_mlp_result['act']}")
    print(f"  Val RMSE:      {best_mlp_result['val_rmse']:.2f}")

    # Save full pipeline (preprocessor + best MLP)
    best_mlp_pipe = Pipeline([('pre', preprocessor), ('model', best_mlp_model)])
    joblib.dump(best_mlp_pipe, f'{MODEL_DIR}/mlp_model.pkl')
    joblib.dump(mlp_results, f'{MODEL_DIR}/mlp_search.pkl')
    print('MLP model saved.')
else:
    mlp_results = joblib.load(f'{MODEL_DIR}/mlp_search.pkl')
    best_mlp_pipe = joblib.load(f'{MODEL_DIR}/mlp_model.pkl')
    best_mlp_result = min(mlp_results, key=lambda r: r['val_rmse'])
    print(f"MLP model loaded. Best val RMSE: {best_mlp_result['val_rmse']:.2f}")

In [ ]:
# Visualisation 3: MLP training loss curve of best model
fig, ax = plt.subplots(figsize=(8, 4))
loss_curve = best_mlp_result['model'].loss_curve_
ax.plot(loss_curve, color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title(f'MLP Training Loss Curve\n(hidden={best_mlp_result["hidden"]}, lr={best_mlp_result["lr"]}, act={best_mlp_result["act"]})')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig3_mlp_loss_curve.png', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## 5. Results & Comparison

In [ ]:
# Evaluate both models on train and validation set
def evaluate(model, X_tr, y_tr, X_v, y_v, name):
    y_tr_pred = model.predict(X_tr)
    y_v_pred  = model.predict(X_v)
    return {
        'Model': name,
        'Train RMSE': np.sqrt(mean_squared_error(y_tr, y_tr_pred)),
        'Val RMSE':   np.sqrt(mean_squared_error(y_v, y_v_pred)),
        'Train R²':   r2_score(y_tr, y_tr_pred),
        'Val R²':     r2_score(y_v, y_v_pred),
    }

results = [
    evaluate(best_ridge, X_train, y_train, X_val, y_val, 'Ridge Regression'),
    evaluate(best_mlp_pipe, X_train, y_train, X_val, y_val, 'MLP Neural Network'),
]

results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.round(2)
print(results_df.to_string())

In [ ]:
# Visualisation 4: Predicted vs Actual for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_to_plot = [
    (best_ridge,    'Ridge Regression'),
    (best_mlp_pipe, 'MLP Neural Network'),
]

for ax, (model, name) in zip(axes, models_to_plot):
    y_pred = model.predict(X_val)
    ax.scatter(y_val, y_pred, alpha=0.15, s=5, color='steelblue')
    mn, mx = y_val.min(), y_val.max()
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
    r2 = r2_score(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    ax.set_title(f'{name}\nR² = {r2:.4f}, RMSE = {rmse:.0f} MW')
    ax.set_xlabel('Actual Power Consumption (MW)')
    ax.set_ylabel('Predicted Power Consumption (MW)')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig4_predicted_vs_actual.png', bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

## 6. Save Final Model

In [ ]:
# Choose best model based on validation RMSE
ridge_val_rmse_final = results_df.loc['Ridge Regression', 'Val RMSE']
mlp_val_rmse_final   = results_df.loc['MLP Neural Network', 'Val RMSE']

if mlp_val_rmse_final < ridge_val_rmse_final:
    best_model = best_mlp_pipe
    best_model_name = 'MLP Neural Network'
else:
    best_model = best_ridge
    best_model_name = 'Ridge Regression'

joblib.dump(best_model, f'{MODEL_DIR}/final_model.pkl')

import os
size_mb = os.path.getsize(f'{MODEL_DIR}/final_model.pkl') / 1e6
print(f'Final model: {best_model_name}')
print(f'Model file size: {size_mb:.2f} MB (limit: 50 MB)')
print('Done!')